In [16]:
import tensorflow as tf
import keras
import numpy as np
import pandas as pd
from customlayer import MyLayerNorm
import matplotlib.pyplot as plt
import keras_tuner as kt
import os
from pathlib import Path
from functools import partial

In [2]:
# Load the dataset
(X_train_all, y_train_all), (X_test, y_test) = keras.datasets.cifar10.load_data()

X_valid, y_valid = X_train_all[-5000:], y_train_all[-5000:]
X_train, y_train = X_train_all[:-5000], y_train_all[:-5000]

In [3]:
X_train = (X_train / 255.)
X_valid = (X_valid / 255.) 
X_test = (X_test / 255.)

In [17]:
DefaultConv2D = partial(tf.keras.layers.Conv2D, kernel_size=3, padding="same", 
                        activation="relu", kernel_initializer="he_normal")
model = tf.keras.Sequential([
    DefaultConv2D(filters=64, kernel_size=7, input_shape=[32, 32, 3]), 
    tf.keras.layers.MaxPool2D(),
    DefaultConv2D(filters=128), 
    DefaultConv2D(filters=128),
    tf.keras.layers.MaxPool2D(),
    DefaultConv2D(filters=256), 
    DefaultConv2D(filters=256),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.LayerNormalization(axis=[1, 2, 3]),
    tf.keras.layers.Flatten(), 
    tf.keras.layers.Dense(units=128, activation="relu", 
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(0.5), 
    tf.keras.layers.Dense(units=64, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=10, activation="softmax")
])                      

/Users/josephhorzempa/anaconda3/envs/math474/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model2 = tf.keras.Sequential([
    DefaultConv2D(filters=64, kernel_size=7, input_shape=[32, 32, 3]), 
    tf.keras.layers.MaxPool2D(),
    DefaultConv2D(filters=128), 
    DefaultConv2D(filters=128),
    tf.keras.layers.MaxPool2D(),
    DefaultConv2D(filters=256), 
    DefaultConv2D(filters=256),
    tf.keras.layers.MaxPool2D(),
    MyLayerNorm(axis=[1, 2, 3]),
    tf.keras.layers.Flatten(), 
    tf.keras.layers.Dense(units=128, activation="relu", 
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(0.5), 
    tf.keras.layers.Dense(units=64, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=10, activation="softmax")
])   

In [18]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=100)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

In [19]:
history = model.fit(
        X_train, y_train, 
        epochs=200,
        validation_data=(X_valid, y_valid), 
        callbacks=[early_stopping_cb],
        batch_size=128
    )

Epoch 1/200
179/352 ━━━━━━━━━━━━━━━━━━━━ 1:04 371ms/step - accuracy: 0.1209 - loss: 2.3742

KeyboardInterrupt: 